# 11 — CLIP: aprendizaje contrastivo y clasificación zero-shot

CLIP relaciona imágenes y textos mediante un preentrenamiento contrastivo sobre pares. Estudiaremos clasificación zero-shot comparando vectores de imágenes con textos que describen las clases, sin entrenar el encoder con nuestras etiquetas. También se propone una cabeza lineal sobre representaciones congeladas.

El estudio está pendiente. Usar embeddings no implica generar contenido, y usar un prompt de CLIP no constituye por sí solo aprendizaje en contexto.

> **Plan pendiente de implementación.** Este cuaderno contiene explicaciones y propuestas. No contiene código ejecutado ni resultados de entrenamiento.

## Vía C · Embeddings, NLP opcional y conexión con GenAI

Esta es nuestra interpretación del bloque de apuntes «word-vector (embedding) → CNN / RNN / Transformer». **Un embedding es una representación vectorial; por sí solo no es GenAI.** Un vector por palabra de [Word2Vec](https://arxiv.org/abs/1301.3781) es una representación estática. Un encoder contextual como [BERT](https://arxiv.org/abs/1810.04805) produce representaciones dependientes del contexto. Ambos pueden alimentar tareas de clasificación sin generar texto.

| Rama didáctica posible sobre `detail_desc` | Relaciones que aprende | Extensión que se podría contrastar |
|---|---|---|
| Tokenización → embedding → pooling/Dense | Resumen de los vectores | Embeddings aleatorios entrenables frente a preentrenados |
| Tokenización → embedding → CNN1D | Patrones locales de tokens | CNN1D sin/con atención añadida |
| Tokenización → embedding → LSTM/GRU | Dependencias secuenciales mediante estados | Último estado/pooling frente a atención sobre estados |
| Tokens + posición → Transformer encoder | Relaciones mediante self-attention | Encoder pequeño frente a encoder preentrenado |

Esta tabla describe **alternativas opcionales**, no cuatro experimentos ya implementados ni un cambio del objetivo visual principal. Un embedding se adapta a la arquitectura; las palabras no pasan a ser píxeles. La imagen puede representarse como secuencia de columnas (06) o de parches (08), con un significado distinto del token textual.

**Conexión multimodal ya prevista.** [CLIP](https://arxiv.org/abs/2103.00020) aprende a alinear representaciones de imágenes y textos con un objetivo contrastivo. Comparar un embedding de imagen con descripciones de clases permite clasificación zero-shot; no equivale a generar texto o imágenes. Es más preciso describir su origen como preentrenamiento contrastivo con supervisión de pares imagen–texto que tratar todo modelo preentrenado como autosupervisado sin matices. Contraste, transferencia, zero-shot e in-context learning son conceptos relacionados, pero usar un prompt de CLIP no constituye por sí solo un experimento de in-context learning.

**Protocolo si se desarrolla NLP.** Heredar los grupos y splits de 02; ajustar vocabulario propio, longitudes y selección de hiperparámetros con train/validation. Los embeddings externos tendrán su procedencia documentada. Usar solo `detail_desc` como entrada textual, sin añadir la columna objetivo; auditar términos que revelen directamente la categoría. Una descripción puede nombrar la prenda: comparar texto con fotografía como si fuera la misma información sería engañoso. Documentar esa ventaja y estudiar una ablación de términos explícitos usando reglas fijadas sin test. Registrar descripciones vacías o repetidas; si textos idénticos cruzan splits, revisar la unidad de agrupación y versionar cualquier nueva partición, sin sobrescribir la visual.

**BERT y autorregresión.** BERT original reconstruye tokens enmascarados con contexto bidireccional; no se usará como ejemplo de generación causal de siguiente token. La eventual generación tipo GPT se discute en 12. La lectura de los apuntes es nuestra; para confirmar el sentido exacto de la flecha haría falta contrastarla con la explicación de clase.

**Qué podemos reutilizar de 02.** Las puntuaciones guardadas para detectar detalles no son los embeddings completos de las imágenes. No bastan para evaluar nuevos textos de las 26 clases. Si no existen embeddings compatibles, esa extracción adicional se presupuestará y activará expresamente; después se guardará para reutilizarla. Como el conjunto se filtró usando CLIP, su evaluación sobre él está condicionada por esa selección y no es una validación independiente del filtro.

## Objetivos

- Explicar el objetivo contrastivo y distinguir preentrenamiento, transferencia, zero-shot y autosupervisión.
- Comparar textos de clase usando train/validation, sin elegirlos con test.
- Guardar embeddings y predicciones para reutilizar el cálculo.
- Comparar zero-shot y cabeza lineal con los modelos supervisados, documentando los datos externos.
- Mantener recuperación imagen-texto y clasificación con detail_desc como ampliaciones opcionales.

## Pasos previstos

1. Documentar el modelo, preprocesamiento y textos candidatos.
2. Comprobar qué resultados existen y presupuestar la extracción que falte antes de activarla.
3. Guardar embeddings por identificador, hash de imagen, modelo y preprocesamiento.
4. Comparar zero-shot y cabeza lineal en validation.
5. Informar la limitación de haber usado CLIP previamente para seleccionar fotografías.

## Entradas, salidas y material de clase

| | |
|---|---|
| **Entradas** | Train/validation de 02; test reservado para el protocolo final; detail_desc opcional |
| **Salidas** | Embeddings, predicciones, métricas zero-shot y cabeza lineal; esquema de conceptos |
| **Material de clase** | `Interacting_with_CLIP.ipynb`, `Notebooks_GuideLines_for_DeepLearnig_Report.pptx` (learning-strategy takeaways), `MLLB_HuggingFace_Intro.pptx`, `PRDL_MLLB_LLMs_and_Aplications.pptx` |